# Dataset_Audition

## Basic information of 3 datasets

In [1]:
## Data loading
import pandas as pd
airline = pd.read_csv('Datasets/airline_delay.csv')
hotel = pd.read_csv('Datasets/hotel_bookings.csv')
train = pd.read_csv('Datasets/train_occupancy.csv')

## Basic information
basic_information = pd.DataFrame({
    'Dataset': ['Airline Delay', 'Hotel Booking', 'Train Occupancy'],
    'Rows': [airline.shape[0], hotel.shape[0], train.shape[0]],
    'Columns': [airline.shape[1], hotel.shape[1], train.shape[1]],
    'Numeric columns': [
        airline.select_dtypes(include='number').shape[1],
        hotel.select_dtypes(include='number').shape[1],
        train.select_dtypes(include='number').shape[1]
    ],
    'Other columns': [
        airline.select_dtypes(exclude='number').shape[1],
        hotel.select_dtypes(exclude='number').shape[1],
        train.select_dtypes(exclude='number').shape[1]
    ]
})

basic_information

,Dataset,Rows,Columns,Numeric columns,Other columns
0,Airline Delay,101000,29,14,15
1,Hotel Booking,119987,32,20,12
2,Train Occupancy,50250,14,2,12


## Airline Delay

In [2]:
## Date range
airline_date = pd.to_datetime(
    airline['Year'].astype(str) + '-' +
    airline['Month'].astype(str) + '-' +
    airline['DayofMonth'].astype(str)
)
print('Date range:', airline_date.min().date(), 'to', airline_date.max().date())
print()

## Duplicate rows
airline_duplicate_rows = airline.duplicated().sum()
print('Duplicate rows:', airline_duplicate_rows)
print()

## Check distribution: delays >=15
valid_arrival_delay = airline['ArrDelay'].dropna()
airline_delayed_count = (valid_arrival_delay >= 15).sum()
airline_delay_rate = airline_delayed_count / len(valid_arrival_delay) * 100
print('Valid ArrDelay values:', len(valid_arrival_delay))
print('Missing ArrDelay values:', airline['ArrDelay'].isna().sum())
print(f'ArrDelay >= 15 minutes: {airline_delayed_count:,} ({airline_delay_rate:.2f}%)')
print()

## Carrier distribution
airline_carrier_share = (
    airline['UniqueCarrier']
    .str.strip()
    .str.upper()
    .value_counts(normalize=True) * 100
)
print('Carrier percentages:')
display(
    airline_carrier_share
    .rename('Percentage')
    .reset_index()
)
print(f"WN share: {airline_carrier_share.loc['WN']:.2f}%")
print()

## Cancelled categories consistency
print('Cancelled categories consistency:')
display(
    airline['Cancelled']
    .value_counts(dropna=False)
    .rename('Count')
    .reset_index()
)
print()

## Potential target data leakage
delay_cols = [
    'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay'
]
print('Potential leakage variables:')
display(
    airline[['ArrDelay'] + delay_cols]
    .dropna()
    .head(5)
)


Date range: 2008-01-01 to 2008-01-31

Duplicate rows: 1000

Valid ArrDelay values: 97700
Missing ArrDelay values: 3300
ArrDelay >= 15 minutes: 19,440 (19.90%)

Carrier percentages:


,UniqueCarrier,Percentage
0,WN,93.581188
1,XE,5.896040
2,W,0.483168
3,X,0.039604


WN share: 93.58%

Cancelled categories consistency:


,Cancelled,Count
0,0,98371
1,1,1632
2,TRUE,262
3,N,251
4,Y,249
5,FALSE,235



Potential leakage variables:


,ArrDelay,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
32,19.0,1.0,0.0,0.0,0.0,18.0
36,24.0,0.0,0.0,24.0,0.0,0.0
44,22.0,3.0,0.0,0.0,0.0,19.0
49,30.0,0.0,0.0,3.0,0.0,27.0
53,57.0,0.0,0.0,0.0,0.0,57.0


## Hotel Booking

In [3]:
## Date range
hotel_arrival_date = pd.to_datetime(
    hotel['arrival_date_year'].astype(str) + '-' +
    hotel['arrival_date_month'] + '-' +
    hotel['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)
print(
    'Arrival date range:',
    hotel_arrival_date.min().date(),
    'to',
    hotel_arrival_date.max().date()
)
print()

## Duplicate rows
hotel_duplicate_rows = hotel.duplicated().sum()
print('Duplicate-looking rows:', hotel_duplicate_rows)
print()

## Check distribution: is_canceled
hotel_cancelled = (hotel['is_canceled'] == 1).sum()
hotel_cancel_rate = hotel_cancelled / len(hotel) * 100
print('Missing is_canceled values:', hotel['is_canceled'].isna().sum())
print(f'Cancelled records: {hotel_cancelled:,} ({hotel_cancel_rate:.2f}%)')
display(
    hotel['is_canceled']
    .value_counts(dropna=False)
    .sort_index()
    .rename('Count')
    .reset_index()
)
print()

## Selected missing values
print('Selected missing values:')
display(
    hotel[['children', 'country', 'agent', 'company']]
    .isna()
    .sum()
    .rename('Missing')
    .reset_index()
    .rename(columns={'index': 'Variable'})
)
print()

## Hotel category consistency
print('Hotel categories consistency:')

display(
    hotel['hotel']
    .value_counts(dropna=False)
    .rename('Count')
    .reset_index()
)
print()

## Potential target data leakage
leakage_cols = [
    'reservation_status',
    'reservation_status_date'
]
print('Potential leakage variables:')
display(
    hotel[['is_canceled'] + leakage_cols]
    .dropna()
    .head(5)
)

Arrival date range: 2015-07-01 to 2017-08-31

Duplicate-looking rows: 31328

Missing is_canceled values: 0
Cancelled records: 44,448 (37.04%)


,is_canceled,Count
0,0,75539
1,1,44448



Selected missing values:


,Variable,Missing
0,children,182
1,country,628
2,agent,16541
3,company,113162



Hotel categories consistency:


,hotel,Count
0,City Hotel,79570
1,Resort Hotel,40188
2,CityHotel,142
3,ResortHotel,87



Potential leakage variables:


,is_canceled,reservation_status,reservation_status_date
0,1,Canceled,2016-01-15
1,0,Check-Out,2017-03-12
2,0,Check-Out,2016-02-28
3,1,Canceled,2017-03-01
4,0,Check-Out,2016-07-01


## Train Occupancy

In [4]:
## Operational day range
print(
    'Operational day range:',
    train['day'].min(),
    'to',
    train['day'].max(),
    f"({train['day'].nunique()} distinct days)"
)
print()

## Duplicate rows
train_duplicate_rows = train.duplicated().sum()
print('Duplicate rows:', train_duplicate_rows)
print()

## Target distribution: Occupancy Range
train_occupancy_summary = (
    train['Occupancy Range']
    .value_counts(dropna=False)
    .rename('Count')
    .reset_index()
)
train_occupancy_summary['Percentage'] = (
    train_occupancy_summary['Count'] / len(train) * 100
).round(2)
display(train_occupancy_summary)
print()

## Missing Occupancy Status values
print(
    'Missing Occupancy Status values:',
    train['Occupancy Status'].isna().sum()
)
print()

## Occupancy Status category consistency
print('Occupancy Status categories:')
display(
    train['Occupancy Status']
    .value_counts(dropna=False)
    .rename('Count')
    .reset_index()
)

Operational day range: 9 to 16 (8 distinct days)

Duplicate rows: 250



,Occupancy Range,Count,Percentage
0,Low: 0-399,45932,91.41
1,Medium: 400-799,3556,7.08
2,High: 800+,762,1.52



Missing Occupancy Status values: 73

Occupancy Status categories:


,Occupancy Status,Count
0,MANY_SEATS_AVAILABLE,48008
1,FEW_SEATS_AVAILABLE,1596
2,STANDING_ROOM_ONLY,398
3,MANY_SEATSAVAILABLE,165
4,NaN,73
5,few_seats_available,8
6,VTANDING_ROOM_ONLY,2


In [5]:
## Values calculation
airline_wn_share = airline_carrier_share.loc['WN']

train_low_rate = train_occupancy_summary.loc[
    train_occupancy_summary['Occupancy Range'] == 'Low: 0-399',
    'Percentage'
].iloc[0]

train_medium_rate = train_occupancy_summary.loc[
    train_occupancy_summary['Occupancy Range'] == 'Medium: 400-799',
    'Percentage'
].iloc[0]

train_high_rate = train_occupancy_summary.loc[
    train_occupancy_summary['Occupancy Range'] == 'High: 800+',
    'Percentage'
].iloc[0]

## Comparison table
comparison = pd.DataFrame({
    'Criterion': [
        'Size',
        'Target',
        'Time coverage',
        'Main limitation',
        'Stage 2 suitability'
    ],

    'Airline Delay': [
        f"{airline.shape[0]:,} × {airline.shape[1]}",
        f"ArrDelay >= 15: {airline_delay_rate:.2f}%",
        f"{airline_date.min().date()} to {airline_date.max().date()}",
        f"WN = {airline_wn_share:.2f}%; missing target; leakage risk",
        'Moderate'
    ],

    'Hotel Booking': [
        f"{hotel.shape[0]:,} × {hotel.shape[1]}",
        f"is_canceled = 1: {hotel_cancel_rate:.2f}%",
        f"{hotel_arrival_date.min().date()} to {hotel_arrival_date.max().date()}",
        'Duplicate-looking rows; missing values; leakage risk',
        'Strongest'
    ],

    'Train Occupancy': [
        f"{train.shape[0]:,} × {train.shape[1]}",
        (
            f"Low {train_low_rate:.2f}%; "
            f"Medium {train_medium_rate:.2f}%; "
            f"High {train_high_rate:.2f}%"
        ),
        f"Day {train['day'].min()} to {train['day'].max()}",
        'Severe class imbalance; short time coverage',
        'Weakest'
    ]
})

display(
    comparison.set_index('Criterion')
)

,Airline Delay,Hotel Booking,Train Occupancy
Criterion,,,
Size,"101,000 × 29","119,987 × 32","50,250 × 14"
Target,ArrDelay >= 15: 19.90%,is_canceled = 1: 37.04%,Low 91.41%; Medium 7.08%; High 1.52%
Time coverage,2008-01-01 to 2008-01-31,2015-07-01 to 2017-08-31,Day 9 to 16
Main limitation,WN = 93.58%; missing target; leakage risk,Duplicate-looking rows; missing values; leakag...,Severe class imbalance; short time coverage
Stage 2 suitability,Moderate,Strongest,Weakest


## Selected dataset: Hotel Booking

Hotel Booking is selected because it provides the best balance of business value and analytical feasibility. It has a complete and reasonably balanced binary target, with 44,448 of 119,987 bookings (37.04%) cancelled, and covers more than two years from July 2015 to August 2017. This gives it broader temporal coverage than the Airline Delay and Train Occupancy datasets.

The dataset also contains useful numeric, categorical and temporal predictors, making it the strongest option for Stage 2. However, `reservation_status` and `reservation_status_date` must be excluded because they contain post-outcome information and may cause target leakage. The 31,328 duplicate-looking rows require careful treatment because no booking ID is available, while missing `agent` and `company` values may represent “not applicable” rather than ordinary missingness.